College Football Data Pull

In [ ]:
# Imports & API Setup
###########################################
import pandas as pd
import numpy as np
import os
import cfbd
from cfbd.rest import ApiException
###########################################

# API Key Setup (Secure)
###########################################
# Set your API key as an environment variable before running.
# Example (Windows): set CFBD_API_KEY=your_api_key_here
API_KEY = os.getenv("CFBD_API_KEY")
if API_KEY is None:
    raise ValueError("CFBD_API_KEY not found. Please set your API key.")
###########################################

# Configure API
###########################################
configuration = cfbd.Configuration()
configuration.api_key['Authorization'] = API_KEY
configuration.api_key_prefix['Authorization'] = 'Bearer'
api_config = cfbd.ApiClient(configuration)
game_apis = cfbd.GamesApi(api_config)
###########################################

# Data Pull (2025 Season)
###########################################
all_team_weeks_stats = []
for wk in range(1, 15):
    games = game_apis.get_game_team_stats(
        year=2025,
        week=wk,
        classification='fbs',
        season_type='regular'
    )
    for g in games:
        game = g.to_dict()
        for t in game['teams']:
            base_info = {
                'season': 2025,
                'week': wk,
                'team': t['team'],
                'game_id': game['id'],
                'conference': t.get('conference'),
                'homeAway': t.get('homeAway'),
                'points': t.get('points')
            }
            # Add all stat categories
            for s in t['stats']:
                base_info[s['category']] = s['stat']
            all_team_weeks_stats.append(base_info)
###########################################

# Convert to DataFrame
###########################################
raw_df = pd.DataFrame(all_team_weeks_stats)
raw_df.head()
print(raw_df.shape)
###########################################